In [ ]:
# read csv
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer


df = pd.read_csv('../Data/post_eda.csv')
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 579182 entries, 0 to 579181
Data columns (total 19 columns):
 #   Column                                                  Non-Null Count   Dtype  
---  ------                                                  --------------   -----  
 0   Sex                                                     579182 non-null  str    
 1   Site recode ICD-O-3/WHO 2008                            579182 non-null  str    
 2   Behavior code ICD-O-3                                   579182 non-null  str    
 3   Histologic Type ICD-O-3                                 579182 non-null  int64  
 4   RX Summ--Surg Prim Site (1998+)                         579182 non-null  int64  
 5   Radiation recode                                        579182 non-null  str    
 6   Chemotherapy recode (yes, no/unk)                       579182 non-null  str    
 7   Time from diagnosis to treatment in days recode         402665 non-null  float64
 8   Survival months                    

Looking at the above, there are some float64 type features that have null values. After splitting the data, I will handle each of these accordingly:
- Remove records where *Survival months* is null
- Impute all of the rest. I will also add a new binary feature for *Time from diagnosis to treatment* and *Time size over time*, representing whether or not the column was a missing value before imputation.

In [2]:
df = df.dropna(subset=["Survival months"])

#create missing value flag features
df['Time from Diagnosis to Treatment Missing'] = np.where(
    df['Time from diagnosis to treatment in days recode'].isna(),
    1, 0
)

df['Tumor Size Over Time Missing'] = np.where(
    df['Tumor Size Over Time Recode (1988+)'].isna(),
    1, 0
)

In [3]:
#Split the data
X = df.drop('Survival months', axis=1)
y = df['Survival months']

X_train, y_train, X_test, y_test = train_test_split(
    X, y,
    train_size=0.2,
    random_state=42,
    stratify=y
)

To impute the missing feature values , I will do the following:
- Replace missing values with the median in the columns *Time from diagnosis to treatment* and *Tumor size over time*

- For *regional nodes positive* and *examined*, I will median impute the values that were previously above 90. My reasoning for doing this is that the categorical values above 90 not stating the number of nodes does not mean that I should set them to 0. Setting them too low or high would give them strong predicitve power, and would alter the predictive power of other records originally holding those values.

- For *regional nodes positive treatment* and *regional nodes examined treatment*, I will simply one hot encode them, with a category to represent the exact number of nodes being known

In [ ]:
# Columns to be median imputed
median_cols = [
    'Time from diagnosis to treatment in days recode',
    'Tumor Size Over Time Recode (1988+)',
    'Regional nodes positive (1988+)',
    'Regional nodes examined (1988+)'
]

# Categorical columns to replace nulls with 1
one_cols = [
    'Regional nodes positive treatment',
    'Regional nodes examined treatment'
]

# Median imputer
median_imputer = SimpleImputer(strategy='median')

X_train[median_cols] = median_imputer.fit_transform(
    X_train[median_cols]
)

X_test[median_cols] = median_imputer.transform(
    X_test[median_cols]
)

# Constant imputer
one_imputer = SimpleImputer(strategy='constant', fill_value=1)

X_train[one_cols] = one_imputer.fit_transform(
    X_train[one_cols]
)

X_test[one_cols] = one_imputer.transform(
    X_test[one_cols]
)

In [ ]:
# divide features into power transform or oneHot, scale other numerical features
oneHot_features = [
    'Sex',
    'Site recode ICD-O-3/WHO 2008',
    'Behavior code ICD-O-3 ',
    'Histologic Type ICD-O-3',
    'RX Summ--Surg Prim Site (1998+)',
    'Radiation recode',
    'Chemotherapy recode (yes, no/unk)',
    'Visceral and Parietal Pleural Invasion Recode (2010+)',
    'SEER Combined Mets at DX-lung (2010+)',
    'Separate Tumor Nodules Ipsilateral Lung Recode (2010+)',
    'Regional nodes positive treatment',
    'Regional nodes examined treatment',
    'Time from Diagnosis to Treatment Missing',
    'Tumor Size Over Time Missing'
]

pt_features = [
    'Time from diagnosis to treatment in days recode',
    'Survival months',
    'Tumor Size Over Time Recode (1988+)',
    'Regional nodes examined (1988+)',
    'Regional nodes positive (1988+)',
]

scale_features = [
    'Year of diagnosis',
    'Age recode with <1 year olds and 90+'
]